# 10 PySpark Processing

Notebook to validate the PySpark batch processing layer for LexCare AI.

Goals:
- create a local Spark session
- load a small set of sample documents
- process them into chunk rows
- inspect the output as a DataFrame
- optionally export the result for later warehouse steps


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / "_bootstrap.py").exists():
    candidates = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent]
    for candidate in candidates:
        if (candidate / "_bootstrap.py").exists():
            NOTEBOOK_DIR = candidate
            break

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _bootstrap import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, RESULTS_DIR

print(PROJECT_ROOT)
print(RAW_DIR)
print(PROCESSED_DIR)
print(OUTPUT_DIR)
print(RESULTS_DIR)

/Users/rodrigue.lawson/VSCode Projects/lexcare-ai
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/output
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/results


## Imports and Spark setup

This notebook expects PySpark to be available in the project environment.


In [2]:
from datetime import UTC, datetime

import pandas as pd

from app.domain.models import DocumentMetadata, LoadedDocument
from app.infrastructure.spark.session import SparkSessionFactory
from app.services.spark_document_processing_service import SparkDocumentProcessingService

spark = SparkSessionFactory.create(
    app_name="LexCare AI Notebook",
    master="local[1]",
)

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 17:44:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.2


## Sample documents

The notebook uses a small in-memory sample so the Spark pipeline can be validated without depending on a full ingestion run.


In [3]:
sample_documents = [
    LoadedDocument(
        source_path="data/raw/gesetze_im_internet/SGB_11.pdf",
        text=(
            "Pflegegrad 3 bedeutet schwere Beeinträchtigungen der Selbständigkeit "
            "oder der Fähigkeiten und wird bei 47,5 bis unter 70 Gesamtpunkten "
            "zugeordnet."
        ),
        metadata=DocumentMetadata(
            source="gesetze-im-internet",
            document_type="law",
            topic="pflegeversicherung",
            created_at=datetime(2024, 1, 1, tzinfo=UTC),
            extra={"filename": "SGB_11.pdf", "page_count": 177},
        ),
    ),
    LoadedDocument(
        source_path="data/raw/gesetze_im_internet/PUEG_RefE_Pflegereform_bf.pdf",
        text=(
            "Der Referentenentwurf zur Unterstützung und Entlastung in der Pflege "
            "enthält Anpassungen zur Pflegeversicherung und zur Digitalisierung."
        ),
        metadata=DocumentMetadata(
            source="gesetze-im-internet",
            document_type="law",
            topic="pflegereform",
            created_at=datetime(2024, 1, 2, tzinfo=UTC),
            extra={"filename": "PUEG_RefE_Pflegereform_bf.pdf", "page_count": 112},
        ),
    ),
]

len(sample_documents)

2

## Process documents with Spark

The service should return a chunk-level DataFrame with preserved source metadata.


In [4]:
service = SparkDocumentProcessingService(
    spark=spark,
    chunk_size=80,
    chunk_overlap=20,
)

chunk_df = service.process_documents(sample_documents)
chunk_df.show(truncate=False)

+-----------------------------+----------------------------------------+-----------+----------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|document_id                  |chunk_id                                |chunk_index|source_path                                               |text                                                                            |metadata_json                                                                                                                                                                                           |
+-----------------------------+----------------------------------------+-----------+----------------------------------------------------------+-----

## Inspect the output as pandas

Useful for quick notebook exploration and validation.


In [5]:
pdf = chunk_df.toPandas()
pdf[[
    "document_id",
    "chunk_id",
    "chunk_index",
    "source_path",
    "text",
    "metadata_json",
]]

,document_id,chunk_id,chunk_index,source_path,text,metadata_json
0,SGB_11_pdf,SGB_11_pdf-chunk-0000,0,data/raw/gesetze_im_internet/SGB_11.pdf,Pflegegrad 3 bedeutet schwere Beeinträchtigung...,"{""source"": ""gesetze-im-internet"", ""document_ty..."
1,SGB_11_pdf,SGB_11_pdf-chunk-0001,1,data/raw/gesetze_im_internet/SGB_11.pdf,"ndigkeit oder der Fähigkeiten und wird bei 47,...","{""source"": ""gesetze-im-internet"", ""document_ty..."
2,SGB_11_pdf,SGB_11_pdf-chunk-0002,2,data/raw/gesetze_im_internet/SGB_11.pdf,Gesamtpunkten zugeordnet.,"{""source"": ""gesetze-im-internet"", ""document_ty..."
3,PUEG_RefE_Pflegereform_bf_pdf,PUEG_RefE_Pflegereform_bf_pdf-chunk-0000,0,data/raw/gesetze_im_internet/PUEG_RefE_Pfleger...,Der Referentenentwurf zur Unterstützung und En...,"{""source"": ""gesetze-im-internet"", ""document_ty..."
4,PUEG_RefE_Pflegereform_bf_pdf,PUEG_RefE_Pflegereform_bf_pdf-chunk-0001,1,data/raw/gesetze_im_internet/PUEG_RefE_Pfleger...,r Pflege enthält Anpassungen zur Pflegeversich...,"{""source"": ""gesetze-im-internet"", ""document_ty..."


## Summary metrics

These checks help confirm that chunking and metadata propagation work as expected.


In [6]:
summary = pdf.groupby("source_path").agg(
    chunk_count=("chunk_id", "count"),
    first_chunk=("chunk_index", "min"),
    last_chunk=("chunk_index", "max"),
)

summary

,chunk_count,first_chunk,last_chunk
source_path,,,
data/raw/gesetze_im_internet/PUEG_RefE_Pflegereform_bf.pdf,2,0,1
data/raw/gesetze_im_internet/SGB_11.pdf,3,0,2


## Optional export

You can export the Spark result for further warehouse experiments.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "spark_chunk_output.csv"

pdf.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/output/spark_chunk_output.csv


## Cleanup

Stop the Spark session when you are done.


In [8]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
